# WRFGS_Fullroom_Colab

Historical experiment source from the internship project. Outputs and personal runtime metadata are cleared. This version is retained for provenance; see the root README for the maintained entry points. Colab Drive paths describe the original environment and require your own matching inputs. Source: `WRFGS_Fullroom_Colab.ipynb`. No fresh GPU execution is claimed.

# WRF-GS+ on Tanisha's fullroom scene

Run this notebook in **Google Colab → Runtime → Change runtime type → T4 GPU**.

Download `WRFGS_Fullroom_Package.zip` from the task alongside this notebook. It contains your actual room, all scripts, and the pinned authors' RF-specific model/renderer. No Blender editing is needed for this first experiment.

**Status:** the numerical checks and CUDA learning smoke test have passed on a Colab T4. See the saved experiment report for the completed run's training and test results. This clean notebook is for reproducing or resuming the experiment; its outputs are intentionally empty.

Goal: learn **normalised directional RF images** for one room, with fixed RX and new TX positions. This is not absolute RSSI, new-room prediction or a reproduction of the paper's reported scores.

The cells contain commands that run the bundled Python scripts. Read `START_HERE.md` for the detailed explanation and assumptions. If any cell fails, stop there and share the full error.

## 1. Confirm the GPU
This must display an NVIDIA GPU. We recommend T4 for the prepared software versions.

In [ ]:
import os, sys, subprocess, json, hashlib, zipfile
from pathlib import Path
def run(cmd, **kwargs):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.update(kwargs.pop('env', {}))
    with subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          env=env, **kwargs) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(f'Command failed (exit {result}). Read the error printed above before continuing.')
run(['nvidia-smi'])

## 2. Mount Drive and upload the package
Approve Google's Drive mount prompt so the dataset and checkpoints persist after disconnection. Then choose **WRFGS_Fullroom_Package.zip** from your computer.

Only this one room is used. Code runs in `/content`; experiment outputs are stored in `MyDrive/WRFGS_fullroom`.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
archive = Path('/content/WRFGS_Fullroom_Package.zip')
saved_package = Path('/content/drive/MyDrive/WRFGS_fullroom/WRFGS_Fullroom_Package.zip')
if saved_package.exists():
    import shutil
    shutil.copy2(saved_package, archive)
else:
    uploaded = files.upload()
    assert 'WRFGS_Fullroom_Package.zip' in uploaded, 'Upload the supplied package ZIP.'
    archive.write_bytes(uploaded['WRFGS_Fullroom_Package.zip'])
    del uploaded
destination = Path('/content').resolve()
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        candidate = (destination / info.filename).resolve()
        assert candidate.is_relative_to(destination / 'wrfgs_one_room'), 'Unexpected archive path'
        assert ((info.external_attr >> 16) & 0o170000) != 0o120000, 'Symlinks are not accepted'
    z.extractall(destination)
ROOT = Path('/content/wrfgs_one_room')
os.chdir(ROOT)
manifest = json.loads((ROOT / 'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['sha256'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, name
RUN_DIR = Path('/content/drive/MyDrive/WRFGS_fullroom')
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ['WRF_RUN_DIR'] = str(RUN_DIR)
print('Verified source commit:', manifest['wrfgsplus_commit'])
print('Persistent outputs:', RUN_DIR)

## 3. Install the isolated environment
This can take several minutes. Keep the complete build output if it fails. The cell checks for a CUDA 12.x compiler, installs Python 3.11 and PyTorch 2.6.0 with CUDA 12.4 in a separate environment, installs Sionna RT 1.2.1, then compiles the **included RF rasteriser**.

Do not substitute the ordinary optical Gaussian rasteriser. A driver supporting CUDA is not the same as an installed CUDA compiler. The code stops if the runtime/compiler is unsuitable.

In [ ]:
run([sys.executable, 'setup_colab.py'])
PY = '/content/wrfgs_env/bin/python'
print('Use this Python for every following experiment command:', PY)

## 4. Run the checks before generating data
The CPU tests check geometry, train/validation/test separation, pixel directions and coherent beamforming. The GPU smoke test uses an example from the authors' supplied data for three updates through their model and renderer. This confirms the code can compute gradients; it is not a full benchmark reproduction.

In [ ]:
run([PY, '-m', 'unittest', 'test_rf_math.py', '-v'])
run([PY, 'verify_runtime.py'])

## 5. Prepare the room and look at the locations
We use your **fullroom_materials.xml** and its ten meshes. The existing material assignments are simplified concrete/wood/metal assumptions. The room uses **Y for height**: do not reinterpret the coordinates as Z-up.

One RX at `[0.85, 1.2, -4.2]` m; 600 TX positions at height Y=1.2 m. The conservative clearance test checks antennas against object bounds. Positions above low furniture may appear inside its shaded floor-plan footprint. This scene has no ceiling mesh.

In [ ]:
run([PY, 'generate_data.py', 'prepare'])
from IPython.display import display, Image, Markdown
display(Image(filename=str(RUN_DIR / 'room_plan.png')))
display(Markdown('**Material assignments:**'))
print((RUN_DIR / 'materials.json').read_text())

## 6. Generate and inspect eight pilot RF images
First the code checks a known direct path in an empty scene against an analytic antenna-array response. This catches a reversed direction or array-order error. It then checks one pilot image using twice as many ray launches.

The simulation uses a 4x4 receiver array at 3.5 GHz, LOS and up to three specular interactions. The script computes **baseband** paths with the carrier phase included, sums the paths coherently, and forms a Bartlett angular-power image. Each target is normalised by its own maximum.

Horizontal axis = the pinned renderer's longitude; vertical axis = latitude. These are directions, not floor positions. The actual pinned CUDA source uses a full-sphere 90x360 mapping. The isotropic planar array can produce mirrored peaks; that is a known array ambiguity.

Inspect these plots before continuing. A passed code check does not replace checking the scene and images.

In [ ]:
run([PY, 'generate_data.py', 'pilot'])
display(Image(filename=str(RUN_DIR / 'pilot_rf_images.png')))
print('LOS check:', (RUN_DIR / 'los_check.json').read_text())
print('Convergence check:', (RUN_DIR / 'convergence.json').read_text())
print('Loaded radio materials:', (RUN_DIR / 'loaded_materials.json').read_text())

## 7. Generate the complete dataset
This cell creates the remaining RF examples. It saves each example to Drive immediately and skips existing ones when rerun. It reserves **420 training / 90 validation / 90 final-test** positions.

Time depends on GPU, ray tracing and Drive I/O. The code reports progress; no fixed runtime is promised. If physical settings need changing, choose a new `RUN_DIR` in step 2 rather than mixing cached examples.

In [ ]:
run([PY, 'generate_data.py', 'all'])
print((RUN_DIR / 'data_ready.json').read_text())

## 8. Train WRF-GS+
The bundled adapter imports the authors' Gaussian model, deformation network and RF CUDA renderer. It starts with 8,000 virtual Gaussian sources, allows bounded density growth, and uses their L1+SSIM objective. It selects the best checkpoint using **validation MAE only**.

`best.pt` and `last.pt` include both model parts and optimiser state. Every 500 updates the run saves to Drive. This cell automatically resumes an existing checkpoint. If it has already reached 10,000, proceed to the validation plot.

In [ ]:
checkpoint = RUN_DIR / 'model' / 'last.pt'
if checkpoint.exists():
    step = int(subprocess.check_output([PY, '-c',
        "import torch; print(torch.load(r'" + str(checkpoint) + "', map_location='cpu', weights_only=False)['iteration'])"], text=True).strip())
else:
    step = 0
if step < 10000:
    cmd = [PY, 'train_room.py', 'train', '--iterations', '10000']
    if checkpoint.exists(): cmd.append('--resume')
    run(cmd)
else:
    print('Checkpoint has already reached iteration', step)

## 9. Inspect validation before the final test
Lower MAE is better; higher SSIM means more similar images. Look for a meaningful trend rather than assuming a particular score is guaranteed.

If validation is still improving and you have time, enable the optional continuation below. If it is flat, unstable or poor, inspect the setup instead of repeatedly extending training. Do not use the final test to choose training settings.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
history_path = RUN_DIR / 'model' / 'history.csv'
if not history_path.exists():
    raise RuntimeError(f'Training history was not created at {history_path}. Check the training cell output and WRF_RUN_DIR. Do not restart data generation or create a blank CSV.')
if history_path.stat().st_size == 0:
    raise RuntimeError('Training history is empty. Check whether training is still running or failed before its first validation.')
history = pd.read_csv(history_path)
if history.empty:
    raise RuntimeError('No validation rows yet. Wait for the first validation/checkpoint or inspect the training error.')
display(history.tail())
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history.iteration, history.val_mae)
axes[0].set(xlabel='Iteration', ylabel='Validation MAE')
axes[1].plot(history.iteration, history.val_ssim)
axes[1].set(xlabel='Iteration', ylabel='Validation SSIM')
plt.show()

In [ ]:
# OPTIONAL: change to True only if you choose to continue based on validation.
CONTINUE_TRAINING = False
if CONTINUE_TRAINING:
    run([PY, 'train_room.py', 'train', '--iterations', '20000', '--resume'])
# The same schedule supports a later extension to 30000; do not exceed it.

## 10. Fresh reload and final test
This starts a separate process, loads the checkpoint chosen by validation, and evaluates **all 90 final-test positions**. It also compares against the nearest training example and the mean training image.

The comparison figure uses predetermined samples, not hand-picked successes. The report will state whether MAE beats both baselines. If it does not, report successful execution with prediction quality still requiring improvement.

In [ ]:
run([PY, 'train_room.py', 'evaluate'])
summary = json.loads((RUN_DIR / 'evaluation' / 'summary.json').read_text())
print(json.dumps(summary, indent=2))
display(Image(filename=str(RUN_DIR / 'evaluation' / 'comparisons.png')))
display(Image(filename=str(RUN_DIR / 'evaluation' / 'validation_curve.png')))
display(Markdown((RUN_DIR / 'evaluation' / 'mentor_report.md').read_text()))

## 11. Save the experiment package
The completed run archive contains your dataset, raw baseband paths, settings, model checkpoints and metrics. The original source package is saved separately beside it so the experiment can be recreated.

The archive can be large. It is saved in Drive; downloading it is optional.

In [ ]:
import shutil
assert (RUN_DIR / 'evaluation' / 'summary.json').exists(), 'Final evaluation has not completed.'
shutil.copy2('/content/WRFGS_Fullroom_Package.zip', RUN_DIR.parent / 'WRFGS_fullroom_source.zip')
result_zip = shutil.make_archive(str(RUN_DIR.parent / 'WRFGS_fullroom_completed_run'), 'zip', root_dir=RUN_DIR)
print('Saved completed run:', result_zip)
# Optional download:
# files.download(result_zip)

## What to tell sir after the run

“I implemented an adaptation of the official WRF-GS+ code on our fullroom scene. I generated Sionna RF spectra for 600 transmitter positions with a fixed receiver, trained on 420 positions, selected the checkpoint using 90 validation positions, and tested on 90 unseen positions. I have reference-versus-predicted RF images, error metrics and inference time.”

Add the **actual numbers** from `evaluation/mentor_report.md`. If the model underperforms the baselines, say so. This experiment concerns normalised simulated directional RF patterns; it does not establish absolute RSSI accuracy, real-world performance, or generalisation to other rooms.

Sources: [authors' WRF-GS+ code](https://github.com/wenchaozheng/WRF-GSplus), commit `e564f3184463717ae08660a8b7232c6318c73ca6`; [paper](https://arxiv.org/abs/2412.04832); [Sionna RT 1.2.1](https://github.com/NVlabs/sionna-rt/tree/v1.2.1). See `START_HERE.md` for assumptions and troubleshooting.